Cell 1: Setup      

In [14]:
import os
import ollama
import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

load_dotenv("../.env")
conn = psycopg.connect(
    host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
)
register_vector(conn)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready


Cell 2: retrieval

In [15]:
def retrieve(query, k=5, year_from=None, year_to=None):
    q_vec = model.encode(query, normalize_embeddings=True)

    where, params = [], [q_vec]
    if year_from is not None:
        where.append("year >= %s"); params.append(year_from)
    if year_to is not None:
        where.append("year <= %s"); params.append(year_to)
    where_sql = ("WHERE " + " AND ".join(where)) if where else ""

    sql = f"""
        SELECT DISTINCT ON (song_id) song_id, title, artist, year, chunk_text,
               1 - (embedding <=> %s) AS similarity
        FROM chunks {where_sql}
        ORDER BY song_id, embedding <=> %s
    """
    rows = conn.execute(sql, params + [q_vec]).fetchall()
    rows.sort(key=lambda r: r[5], reverse=True)
    return rows[:k]

Cell 3: generation

In [16]:
def ask(question, k=5, year_from=None, year_to=None):
    hits = retrieve(question, k, year_from, year_to)

    context = "\n\n".join(
        f'[{i+1}] "{title}" by {artist} ({year})\n{text}'
        for i, (_, title, artist, year, text, _) in enumerate(hits)
    )

    system = (
        "You answer questions about song lyrics using ONLY the lyric excerpts provided. "
        "Cite songs by title, artist, and year. Paraphrase instead of quoting lyrics at length. "
        "If the excerpts don't contain enough information, say so instead of guessing."
    )
    user = f"Lyric excerpts:\n\n{context}\n\nQuestion: {question}"

    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        options={"temperature": 0.2, "num_ctx": 4096},
    )
    return response["message"]["content"], hits

Cell 4: Testing

In [17]:
answer, hits = ask("What are songs about missing someone who is far away mostly about?")
print(answer)
print("\nSources:")
for _, title, artist, year, _, sim in hits:
    print(f"- {title} - {artist} ({year})  [{sim:.2f}]")

Based on the provided lyric excerpts, songs about missing someone who is far away are mostly about the pain and longing that comes with being separated from a loved one. They express a sense of emptiness and yearning, with the speaker struggling to cope with the absence of the person they miss. The lyrics often convey a sense of desperation and helplessness, with the speaker unable to move on or find closure.

Sources:
- Missing You - john waite (1984)  [0.57]
- Miss Independent - kelly clarkson (2003)  [0.51]
- Missing You - case (2001)  [0.51]
- Say Hey (I Love You) (feat. Cherine Tanya Anderson) - michael franti spearhead, cherine tanya anderson (2009)  [0.50]
- Missing - 2013 Remaster - everything but the girl (1996)  [0.49]
